# Moduł 2/3 — trening CSP + LDA w procedurze LOSO

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform

import joblib
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import sklearn
from scipy import stats
from IPython.display import display
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# ===== KONFIGURACJA MODELU =====
DATA_BASE = Path.cwd() / "dane_eegmmidb"
MANIFEST_PATH = None   # None → zestaw z modułu 1;
RESULTS_BASE = Path.cwd() / "wyniki_modeli" / "CSP_LDA"

N_CSP = 4
TUNE_THRESHOLD = False
DEFAULT_THRESHOLD = 0.50
MIN_VALIDATION_RECALL = 0.20
MIN_VALIDATION_POSITIVES = 3
REFIT_ON_ALL_NON_TEST = True
SAVE_MODELS = True
VERIFY_FILE_HASHES = True

# ===== PREZENTACJA WYNIKÓW PO TRENINGU =====
SHOW_PLOTS = True 
SAVE_PLOTS = True   
TEST_SUBJECT_TO_SHOW = None  
N_SAMPLES = 10

assert N_CSP >= 2
assert N_SAMPLES >= 1
assert 0 < DEFAULT_THRESHOLD < 1
assert 0 <= MIN_VALIDATION_RECALL <= 1
assert MIN_VALIDATION_POSITIVES >= 1
print("Python:", platform.python_version(), "| MNE:", mne.__version__,
      "| scikit-learn:", sklearn.__version__)

## 1. Odczyt gotowych danych i kontroli niezmienności

In [ ]:
def read_json(path):
    with Path(path).open(encoding="utf-8") as f:
        return json.load(f)


def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def stable_hash(payload, size=12):
    return hashlib.sha256(json.dumps(payload, sort_keys=True,
                                     ensure_ascii=False).encode("utf-8")).hexdigest()[:size]


if MANIFEST_PATH is None:
    pointer = read_json(DATA_BASE / "ostatni_zestaw.json")
    manifest_path = (DATA_BASE / pointer["manifest_relative_path"]).resolve()
else:
    manifest_path = Path(MANIFEST_PATH).expanduser().resolve()
manifest = read_json(manifest_path)
if manifest.get("schema_version") != "1.0":
    raise ValueError("Nieobsługiwana wersja struktury danych.")
dataset_dir = manifest_path.parent
for key, info in manifest["files"].items():
    path = dataset_dir / info["path"]
    if not path.is_file():
        raise FileNotFoundError(path)
    if VERIFY_FILE_HASHES and sha256_file(path) != info["sha256"]:
        raise ValueError(f"Niezgodny SHA-256 (zmieniony plik): {path}")

X_all = np.load(dataset_dir / manifest["files"]["X"]["path"], mmap_mode="r")
y_all = np.load(dataset_dir / manifest["files"]["y"]["path"], mmap_mode="r")
groups = np.load(dataset_dir / manifest["files"]["groups"]["path"], mmap_mode="r")
epoch_run = np.load(dataset_dir / manifest["files"]["runs"]["path"], mmap_mode="r")
splits = read_json(dataset_dir / manifest["files"]["splits"]["path"])
SUBJECT_IDS = [int(s) for s in manifest["subject_ids"]]
CLASS_NAMES = {int(k): v for k, v in manifest["labels"].items()}
POSITIVE_LABEL = int(manifest["positive_label"])

if X_all.dtype != np.float32 or X_all.ndim != 3 or list(X_all.shape) != manifest["X_shape"]:
    raise ValueError("Niezgodny format X: spodziewano się float32 i [epoki,kanały,próbki].")
if not (len(X_all) == len(y_all) == len(groups) == len(epoch_run)):
    raise ValueError("Liczby epok w plikach różnią się.")
if y_all.dtype != np.int8 or groups.dtype != np.int16 or epoch_run.dtype != np.int8:
    raise ValueError("Niezgodne typy y/groups/runs.")
if np.unique(groups).tolist() != SUBJECT_IDS or set(np.unique(y_all)) != {1, 2}:
    raise ValueError("Niezgodne identyfikatory/etykiety uczestników.")
if POSITIVE_LABEL != 2 or N_CSP > X_all.shape[1] - 1:
    raise ValueError("T2 musi mieć etykietę 2, a N_CSP <= (liczba kanałów - 1) przy average reference.")
if splits["dataset_id"] != manifest["dataset_id"]:
    raise ValueError("Podziały LOSO pochodzą z innego zestawu.")
if [f["test_subject"] for f in splits["folds"]] != SUBJECT_IDS:
    raise ValueError("Podziały LOSO nie obejmują dokładnie wszystkich wybranych osób.")
for fold in splits["folds"]:
    a, b, c = (set(fold[k]) for k in
               ("train_subjects_internal", "validation_subjects", "test_subjects"))
    if a & b or a & c or b & c or a | b | c != set(SUBJECT_IDS):
        raise ValueError(f"Nieszczelny/niekompletny split LOSO: {fold['fold']}")
    if c != {fold["test_subject"]}:
        raise ValueError("Split zawiera nieprawidłową osobę testową.")

print("Zestaw:", manifest["dataset_id"], "| X:", X_all.shape, "| osób:", len(SUBJECT_IDS))
print("Klasy:", CLASS_NAMES, "| rodzaje zadań:", manifest["task_types"])
print("Protokół LOSO:", len(splits["folds"]), "foldów, seed:", splits["seed"])

## 2. Model CSP + LDA i polityka progu T2

In [ ]:
def make_model():
    return Pipeline([
        ("ToFloat64", FunctionTransformer(
            np.asarray, validate=False, kw_args={"dtype": np.float64}
        )),
        ("CSP", CSP(n_components=N_CSP, reg=None, log=True,
                    norm_trace=False, cov_est="concat",
                    rank={"eeg": X_all.shape[1] - 1})),
        ("LDA", LinearDiscriminantAnalysis()),
    ])


def positive_probabilities(model, X):
    classes = model.classes_
    if POSITIVE_LABEL not in classes:
        raise ValueError("Brak klasy dodatniej w wytrenowanym modelu.")
    column = int(np.flatnonzero(classes == POSITIVE_LABEL)[0])
    return model.predict_proba(X)[:, column]


def labels_from_probability(prob, threshold):
    return np.where(prob >= threshold, POSITIVE_LABEL, 1).astype(np.int8)


def choose_validation_threshold(y_true, prob):
    def scores(threshold):
        pred = labels_from_probability(prob, threshold)
        return (float(precision_score(y_true, pred, pos_label=2, zero_division=0)),
                float(recall_score(y_true, pred, pos_label=2, zero_division=0)))

    if not TUNE_THRESHOLD:
        p, r = scores(DEFAULT_THRESHOLD)
        return float(DEFAULT_THRESHOLD), "fixed", p, r
    candidates = []
    for threshold in np.linspace(0.05, 0.95, 91):
        predicted = labels_from_probability(prob, threshold)
        n_positive = int((predicted == POSITIVE_LABEL).sum())
        p, r = scores(threshold)
        if r >= MIN_VALIDATION_RECALL and n_positive >= MIN_VALIDATION_POSITIVES:
            candidates.append((p, r, -abs(threshold - 0.5), float(threshold)))
    if not candidates:
        p, r = scores(DEFAULT_THRESHOLD)
        return float(DEFAULT_THRESHOLD), "fallback_0.5", p, r
    p, r, _, threshold = max(candidates)
    return threshold, "validation_optimized", p, r


def calculate_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[1, 2]).ravel()
    return {
        "precision_T2": float(precision_score(y_true, y_pred, pos_label=2, zero_division=0)),
        "recall_T2": float(recall_score(y_true, y_pred, pos_label=2, zero_division=0)),
        "f1_T2": float(f1_score(y_true, y_pred, pos_label=2, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "n_test_epochs": int(len(y_true)),
        "n_predicted_T2": int((y_pred == 2).sum()),
    }

## 3. Wyniki, modele i konfiguracja eksperymentu

In [ ]:
TRAIN_CONFIG = {
    "model": "CSP_LDA", "dataset_id": manifest["dataset_id"],
    "n_csp": N_CSP,
    "csp_reg": None, "csp_log": True, "csp_norm_trace": False,
    "csp_cov_est": "concat", "csp_rank": X_all.shape[1] - 1,
    "tune_threshold": TUNE_THRESHOLD, "default_threshold": DEFAULT_THRESHOLD,
    "min_validation_recall": MIN_VALIDATION_RECALL,
    "min_validation_positives": MIN_VALIDATION_POSITIVES,
    "refit_on_all_non_test": REFIT_ON_ALL_NON_TEST,
    "positive_label": POSITIVE_LABEL, "save_models": SAVE_MODELS,
    "versions": {"python": platform.python_version(), "mne": mne.__version__,
                 "sklearn": sklearn.__version__, "numpy": np.__version__},
}
TRAIN_HASH = stable_hash(TRAIN_CONFIG)
OUTPUT_DIR = RESULTS_BASE / manifest["dataset_id"] / TRAIN_HASH
MODEL_DIR = OUTPUT_DIR / "modele"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if SAVE_MODELS:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
experiment_manifest = {
    "experiment_id": TRAIN_HASH, "model": "CSP_LDA",
    "dataset_id": manifest["dataset_id"],
    "data_manifest_relative_path": os.path.relpath(manifest_path, OUTPUT_DIR),
    "config": TRAIN_CONFIG, "n_folds_expected": len(SUBJECT_IDS),
    "model_file_pattern": "modele/S{subject:03d}.joblib" if SAVE_MODELS else None,
    "results": {"per_subject": "wyniki_per_subject.csv",
                "predictions": "predykcje_epok.csv",
                "assignments": "podzialy_uczestnikow.csv"},
}
with (OUTPUT_DIR / "manifest_modelu.json").open("w", encoding="utf-8") as f:
    json.dump(experiment_manifest, f, indent=2, ensure_ascii=False)

print("Wyniki eksperymentu:", OUTPUT_DIR)
print("Modele na dysku:", SAVE_MODELS)

## 4. Główna pętla LOSO — trening i predykcje poza próbą

In [ ]:
fold_results, fold_assignments, predictions = [], [], []
for fold in splits["folds"]:
    fold_no = int(fold["fold"])
    test_subject = int(fold["test_subject"])
    train_people = fold["train_subjects_internal"]
    val_people = fold["validation_subjects"]

    train_idx = np.flatnonzero(np.isin(groups, train_people))
    val_idx = np.flatnonzero(np.isin(groups, val_people))
    test_idx = np.flatnonzero(groups == test_subject)
    non_test_idx = np.flatnonzero(groups != test_subject)
    if len(train_idx) + len(val_idx) + len(test_idx) != len(groups):
        raise AssertionError(f"Fold {fold_no}: niepełny podział epok")
    if any(set(np.unique(y_all[ix])) != {1, 2} for ix in (train_idx, val_idx, test_idx)):
        raise ValueError(f"Fold {fold_no}: brak obu klas w jednym z podziałów")

    validation_model = make_model()
    validation_model.fit(X_all[train_idx], y_all[train_idx])
    validation_prob = positive_probabilities(validation_model, X_all[val_idx])
    threshold, method, val_precision, val_recall = choose_validation_threshold(
        y_all[val_idx], validation_prob
    )

    if REFIT_ON_ALL_NON_TEST:
        final_model = make_model()
        final_model.fit(X_all[non_test_idx], y_all[non_test_idx])
    else:
        final_model = validation_model

    test_prob = positive_probabilities(final_model, X_all[test_idx])
    test_pred = labels_from_probability(test_prob, threshold)
    metrics = calculate_metrics(y_all[test_idx], test_pred)

    if SAVE_MODELS:
        joblib.dump(final_model, MODEL_DIR / f"S{test_subject:03d}.joblib", compress=3)

    fold_results.append({
        "fold": fold_no, "test_subject": test_subject,
        "dataset_id": manifest["dataset_id"],
        "n_train_subjects_internal": len(train_people),
        "n_validation_subjects": len(val_people),
        "n_train_subjects_final": len(SUBJECT_IDS) - 1
            if REFIT_ON_ALL_NON_TEST else len(train_people),
        "n_train_epochs_internal": int(len(train_idx)),
        "n_validation_epochs": int(len(val_idx)),
        "threshold_T2": threshold, "threshold_method": method,
        "validation_precision_T2": val_precision,
        "validation_recall_T2": val_recall,
        **metrics,
    })
    fold_assignments.append({
        "fold": fold_no, "test_subject": test_subject,
        "train_subjects_internal": ",".join(map(str, train_people)),
        "validation_subjects": ",".join(map(str, val_people)),
        "test_subjects": str(test_subject),
        "final_train_subjects": ",".join(
            map(str, [s for s in SUBJECT_IDS if s != test_subject]
                if REFIT_ON_ALL_NON_TEST else train_people)
        ),
    })
    for test_epoch, (global_index, pred, prob) in enumerate(
        zip(test_idx, test_pred, test_prob), start=1
    ):
        predictions.append({
            "fold": fold_no, "test_subject": test_subject,
            "test_epoch": test_epoch, "global_epoch_index": int(global_index),
            "run": int(epoch_run[global_index]),
            "y_true": int(y_all[global_index]), "y_pred": int(pred),
            "probability_T2": float(prob), "threshold_T2": float(threshold),
        })

    pd.DataFrame(fold_results).to_csv(OUTPUT_DIR / "wyniki_per_subject.csv", index=False)
    pd.DataFrame(fold_assignments).to_csv(
        OUTPUT_DIR / "podzialy_uczestnikow.csv", index=False
    )
    pd.DataFrame(predictions).to_csv(OUTPUT_DIR / "predykcje_epok.csv", index=False)

    print(f"[{fold_no:03d}/{len(SUBJECT_IDS):03d}] test S{test_subject:03d} | "
          f"precision T2={metrics['precision_T2']:.3f} | "
          f"recall T2={metrics['recall_T2']:.3f} | "
          f"balanced accuracy={metrics['balanced_accuracy']:.3f}")

results_df = pd.DataFrame(fold_results)
predictions_df = pd.DataFrame(predictions)
assert results_df["test_subject"].tolist() == SUBJECT_IDS
assert len(predictions_df) == len(X_all)
assert set(predictions_df["global_epoch_index"]) == set(range(len(X_all)))

# Zapis wskaźnika dopiero po zakończeniu WSZYSTKICH foldów.
RESULTS_BASE.mkdir(parents=True, exist_ok=True)
with (RESULTS_BASE / "ostatni_eksperyment.json").open("w", encoding="utf-8") as f:
    json.dump({"experiment_relative_path": str(OUTPUT_DIR.relative_to(RESULTS_BASE)),
               "dataset_id": manifest["dataset_id"],
               "experiment_id": TRAIN_HASH}, f, indent=2)
print("\nZakończono LOSO:", len(results_df), "osób,", len(predictions_df), "predykcji")
print("Wyniki i modele:", OUTPUT_DIR)

## 5. Wyniki treningu — tabela i statystyki LOSO

In [ ]:
METRIC_COLS = ["precision_T2", "recall_T2", "f1_T2", "balanced_accuracy", "accuracy"]
summary_rows = []
for metric in METRIC_COLS:
    values = results_df[metric].to_numpy(dtype=float)
    n = len(values)
    mean = float(np.mean(values))
    sd = float(np.std(values, ddof=1)) if n > 1 else 0.0
    if n <= 1 or np.isclose(sd, 0.0):
        ci_low = ci_high = mean
    else:
        ci_low, ci_high = stats.t.interval(
            0.95, df=n - 1, loc=mean, scale=stats.sem(values)
        )
    summary_rows.append({
        "metric": metric, "n_subjects": n,
        "mean": mean, "std": sd,
        "CI95_lower": float(ci_low), "CI95_upper": float(ci_high),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "podsumowanie_loso.csv", index=False)

print("Wyniki dla wszystkich uczestników testowych:")
display(results_df[[
    "test_subject", "precision_T2", "recall_T2", "f1_T2",
    "balanced_accuracy", "accuracy", "n_test_epochs",
    "n_predicted_T2", "threshold_T2",
]].round(3))
print("Podsumowanie LOSO (każda osoba ma równą wagę):")
display(summary_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for metric, label in [
    ("precision_T2", "Precision T2"),
    ("recall_T2", "Recall T2"),
    ("balanced_accuracy", "Balanced accuracy"),
]:
    axes[0].plot(results_df["test_subject"], results_df[metric],
                 marker="o", markersize=3, linewidth=1.0, label=label)
axes[0].set(title="Wyniki LOSO dla każdej osoby", xlabel="Osoba testowa",
            ylabel="Wartość metryki", ylim=(0, 1.05))
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize="small")

axes[1].boxplot([results_df[metric].to_numpy() for metric in METRIC_COLS])
axes[1].set_xticks(range(1, len(METRIC_COLS) + 1))
axes[1].set_xticklabels(METRIC_COLS, rotation=35, ha="right")
axes[1].set(title="Rozkład metryk między uczestnikami", ylabel="Wartość metryki",
            ylim=(0, 1.05))
axes[1].grid(axis="y", alpha=0.25)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(OUTPUT_DIR / "wyniki_loso.png", dpi=160, bbox_inches="tight")
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)


## 6. Macierz pomyłek i raport klasyfikacji

In [ ]:
y_true_all = predictions_df["y_true"].to_numpy(dtype=np.int8)
y_pred_all = predictions_df["y_pred"].to_numpy(dtype=np.int8)
report_text = classification_report(
    y_true_all, y_pred_all, labels=[1, 2],
    target_names=[CLASS_NAMES[1], CLASS_NAMES[2]], zero_division=0,
)
print("Raport klasyfikacji (agregacja po epokach):")
print(report_text)
(OUTPUT_DIR / "raport_klasyfikacji.txt").write_text(report_text, encoding="utf-8")

cm = confusion_matrix(y_true_all, y_pred_all, labels=[1, 2])
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[CLASS_NAMES[1], CLASS_NAMES[2]],
).plot(values_format="d", cmap="Blues", ax=ax, colorbar=False)
ax.set_title(f"CSP + LDA: macierz pomyłek LOSO ({len(SUBJECT_IDS)} osób)")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(OUTPUT_DIR / "macierz_pomylek.png", dpi=160, bbox_inches="tight")
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)


## 7. Przykładowe predykcje wytrenowanego modelu

In [ ]:
subject_to_show = (SUBJECT_IDS[-1] if TEST_SUBJECT_TO_SHOW is None
                   else int(TEST_SUBJECT_TO_SHOW))
if subject_to_show not in SUBJECT_IDS:
    raise ValueError(f"S{subject_to_show:03d} nie występuje w wynikach LOSO.")
chosen = (predictions_df[predictions_df["test_subject"] == subject_to_show]
          .sort_values("test_epoch").reset_index(drop=True))
print(f"Przykładowe predykcje testowe S{subject_to_show:03d}: "
      f"{min(N_SAMPLES, len(chosen))} z {len(chosen)} epok")
examples = chosen[[
    "test_epoch", "run", "y_true", "y_pred", "probability_T2", "threshold_T2",
]].head(N_SAMPLES).copy()
examples["prawda"] = examples["y_true"].map(CLASS_NAMES)
examples["predykcja"] = examples["y_pred"].map(CLASS_NAMES)
examples["poprawnie"] = examples["y_true"].eq(examples["y_pred"])
display(examples[[
    "test_epoch", "run", "prawda", "predykcja",
    "probability_T2", "threshold_T2", "poprawnie",
]].round(3))

print("\nZapisane wyniki eksperymentu:", OUTPUT_DIR)
print("- wyniki_per_subject.csv, predykcje_epok.csv, podzialy_uczestnikow.csv")
print("- podsumowanie_loso.csv, raport_klasyfikacji.txt")
if SAVE_PLOTS:
    print("- wyniki_loso.png, macierz_pomylek.png")
if SAVE_MODELS:
    print("- modele/Sxxx.joblib (osobny model dla każdego folda LOSO)")
